# Stage 6 — Instruction tuning

**Goal:** turn a text completer into something that responds to a request.

Right now the model continues text. Given "Once upon a time" it writes a story;
given "Write a story about a puppy" it will happily continue *that sentence*
rather than obey it. Nothing has taught it that some text is an instruction.

### The two ideas

**Chat template.** Turns get wrapped in ChatML so the model can distinguish "what
I was asked" from "what I should say":

```
<|im_start|>user
Write a story about a lost puppy.<|im_end|>
<|im_start|>assistant
Once upon a time...<|im_end|>
```

The same template string is written into `tokenizer_config.json`, from there into
the GGUF metadata, and from there `llama-server` uses it to format incoming API
requests. **One string, four hops.** If it disagrees anywhere along that chain,
the served model receives prompts in a format it was never trained on and quietly
gets worse for no visible reason.

**Completion-only loss masking.** Prompt tokens get label `-100` so they
contribute no gradient. Without it, the model spends capacity learning to
*generate instructions*, which is not the job.

This one fails **silently** — training still converges, just to a worse model. So
we decode a real batch below and assert the mask is exactly where it should be.

### The dataset has a shape you have to handle

`TinyStoriesInstruct` is stored **one line per row**, not one example per row.
Records are separated by `<|endoftext|>`, and the header fields appear in
**arbitrary order**:

```
Features: Dialogue                 Summary: Lily steals a bike...
Words: quit, oak, gloomy           Words: ride, work, upset
Summary: Sara and Ben...           Features: Dialogue, BadEnding
Story:                             Story:
<blank>                            <blank>
Sara and Ben were playing...       Lily liked to ride her bike...
<|endoftext|>                      <|endoftext|>
```

So `sft.iter_instruct_records` accumulates lines into records and matches headers
by prefix rather than position. Treating rows as examples — the obvious thing —
would yield garbage.

In [ ]:
# --- Colab bootstrap -------------------------------------------------------
# Set this to YOUR GitHub repo once; every notebook uses the same cell.
REPO_URL = "https://github.com/pythonstudentiam/e2e_llm_demo.git"

import os, subprocess, sys
from pathlib import Path

REPO = Path("/content/e2e_llm_demo")
WORK = Path("/content/work")          # scratch: data + checkpoints (ephemeral!)
WORK.mkdir(parents=True, exist_ok=True)

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

sys.path.insert(0, str(REPO / "src"))

# Colab ships torch; these are the rest. -q to keep the log readable.
%pip install -q sentencepiece "datasets>=3.0" "transformers>=4.45" "huggingface_hub>=0.30"

# HF token from the Colab Secrets panel (key icon, left sidebar). Name it
# HF_TOKEN and enable Notebook access -- the grant is PER NOTEBOOK, so every
# notebook asks separately. Never paste a token into a cell.
#
# login() rather than just setting the env var: it writes the token where every
# huggingface_hub call looks, including ones that ignore the environment.
try:
    from google.colab import userdata
    from huggingface_hub import login

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF_TOKEN loaded from Colab Secrets, authenticated")
except Exception as e:
    print("=" * 72)
    print(f"  HF_TOKEN IS NOT AVAILABLE  ({type(e).__name__}: {e})")
    print()
    print("  Every Hub call in this notebook will fail with 401 Unauthorized.")
    print("  Fix: click the key icon in the left sidebar, turn on Notebook")
    print("       access for HF_TOKEN, then RE-RUN THIS CELL before continuing.")
    print("=" * 72)

import torch
print(f"torch {torch.__version__} | CUDA {torch.cuda.is_available()} | "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

In [ ]:
# A T4 is expected. Anything without CUDA means Runtime > Change runtime type > T4 GPU.
import torch
assert torch.cuda.is_available(), (
    "No GPU. Runtime > Change runtime type > Hardware accelerator: T4 GPU, then re-run."
)
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name} | compute capability {cap[0]}.{cap[1]} | {mem:.1f} GB")

# Turing (7.5) has no bf16. That is why training uses fp16 + GradScaler.
if cap[0] < 8:
    print("\n-> Pre-Ampere GPU: bf16 unavailable, fp16 autocast + loss scaling it is.")
else:
    print("\n-> Ampere or newer: bf16 would work here and would let you drop the GradScaler.")

In [ ]:
from tinyllm import config
from tinyllm.config import (
    model_cfg, train_cfg, data_cfg, tok_cfg, sft_cfg, gen_cfg, quant_cfg, serve_cfg, hub,
)

print(config.summary())

In [ ]:
from pathlib import Path
import torch, gc, numpy as np
from huggingface_hub import hf_hub_download
from tinyllm.tokenizer import load_sp
from tinyllm.data import load_tokens, ensure_token_file
from tinyllm.train import build_model, load_checkpoint, pull_checkpoint

data_dir, tok_dir = WORK / "data", WORK / "tokenizer"
sp_path = tok_dir / "tokenizer.model"

if not sp_path.exists():
    tok_dir.mkdir(parents=True, exist_ok=True)
    got = hf_hub_download(repo_id=hub.ckpt_repo, filename="tokenizer/tokenizer.model")
    sp_path.write_bytes(Path(got).read_bytes())
sp = load_sp(sp_path)

ckpt = WORK / "checkpoints/latest.pt"
if not ckpt.exists():
    print("pulling the pretrained checkpoint from the Hub...")
    ckpt = pull_checkpoint(local_dir=WORK / "checkpoints")
assert ckpt and Path(ckpt).exists(), "No checkpoint found -- run notebook 04 first."

model = build_model(model_cfg)
step, meta = load_checkpoint(Path(ckpt), model)
print(f"loaded checkpoint from step {step:,}")

# Local, else the Hub, else rebuild it from the dataset (~1 min for val) and
# upload so the next runtime does not have to. Colab recycles runtimes, so
# this file is absent at the start of most sessions.
val_tokens = load_tokens(ensure_token_file("val.bin", data_dir, hub.ckpt_repo, sp=sp))
print(f"val tokens: {len(val_tokens):,}")

## 6.1 — Baseline: watch it fail to follow an instruction

Before changing anything, confirm the problem exists.

In [ ]:
from tinyllm.evaluate import generate

for instr in gen_cfg.eval_instructions:
    print(f"INSTRUCTION: {instr}")
    print(f"BASE MODEL:  {generate(model, sp, instr, max_new_tokens=100, seed=0)}")
    print("-" * 78)

It continues the instruction as if it were prose, rather than carrying it out.
That is exactly what a base model does, and exactly what this stage fixes.

## 6.2 — Parse the instruct dataset

In [ ]:
from tinyllm.sft import stream_instruct_records, build_instruction
import numpy as np

rng = np.random.default_rng(0)
for i, rec in enumerate(stream_instruct_records(limit=3)):
    print(f"--- record {i} ---")
    print(f"  features : {rec.get('features')}")
    print(f"  words    : {rec.get('words')}")
    print(f"  summary  : {rec.get('summary', '')[:80]}...")
    print(f"  story    : {rec['story'][:80]}...")
    print(f"  -> instruction: {build_instruction(rec, rng)}")
    print()

Instructions are synthesised from the metadata using **sampled templates** rather
than one fixed phrasing. A single template produces a model that breaks the
moment a user words the request differently — it learns the sentence, not the task.

## 6.3 — The stage 6 gate: prove the mask is right

Decode one example and show exactly what the loss does and does not see. This is
a thirty-second check that catches a bug which is otherwise invisible.

In [ ]:
from tinyllm.sft import encode_example, describe_masking

rec = next(stream_instruct_records(limit=1))
ex = encode_example(sp, build_instruction(rec, rng), rec["story"])
report = describe_masking(sp, ex)   # raises if the mask is wrong

print(f"total tokens      {report['n_tokens']}")
print(f"masked (no loss)  {report['n_masked']}")
print(f"supervised        {report['n_supervised']}")
print()
print("MASKED -- the model reads this but is never scored on it:")
print(f"  {report['masked_text']!r}")
print()
print("SUPERVISED -- this is what it is actually trained to produce:")
print(f"  {report['supervised_text'][:300]!r}")

In [ ]:
# Visualise the boundary token by token.
ids, labels = ex["input_ids"], ex["labels"]
print("first 40 tokens (X = masked, . = supervised):\n")
for i in range(min(40, len(ids))):
    flag = "X" if labels[i] == sft_cfg.ignore_index else "."
    print(f"  {flag} {sp.IdToPiece(ids[i])!r}")

The `X`s stop exactly where the assistant's turn begins. That is the whole
invariant.

## 6.4 — Build the dataset

Examples longer than the context are **dropped, not truncated**. Truncating would
cut stories off mid-sentence and teach the model to stop arbitrarily — a subtle
way to make output worse that's hard to trace back later.

In [ ]:
from tinyllm.sft import build_sft_dataset

sft_data = build_sft_dataset(sp, limit=80_000)

lens = [len(e["input_ids"]) for e in sft_data]
print(f"\n  mean length   {np.mean(lens):.0f} tokens")
print(f"  max length    {np.max(lens)} tokens")
print(f"  mean masked   {np.mean([e['n_prompt'] for e in sft_data]):.0f} tokens "
      f"({np.mean([e['n_prompt'] / len(e['input_ids']) for e in sft_data]):.0%} of each example)")

## 6.5 — Fine-tune

~1,500 steps, about 10 minutes. The learning rate is **6× below pretraining**:
this stage is meant to reshape output format, not relearn language. Too high an
LR here erases pretrained knowledge and the model gets worse at the one thing it
was good at.

In [ ]:
from tinyllm.sft import train_sft

model, sft_history = train_sft(model, sp, sft_data, out_dir=WORK / "checkpoints/sft")

## 6.6 — Before and after

Same instructions as section 6.1.

In [ ]:
from tinyllm.sft import chat

for instr in gen_cfg.eval_instructions:
    print(f"INSTRUCTION: {instr}")
    print(f"SFT MODEL:   {chat(model, sp, instr, max_new_tokens=200)}")
    print("-" * 78)

## 6.7 — The metric that gets worse, and why that's correct

Re-measure perplexity on the *pretraining* distribution.

In [ ]:
from tinyllm.evaluate import perplexity, compare
import json

sft_ppl = perplexity(model, val_tokens, n_batches=100)
base = json.loads((WORK / "reports/base_eval.json").read_text())

compare(
    {"val_loss": base["val_loss"], "val_perplexity": base["val_perplexity"],
     "bits_per_token": base["bits_per_token"]},
    {"val_loss": sft_ppl["loss"], "val_perplexity": sft_ppl["perplexity"],
     "bits_per_token": sft_ppl["loss"] / np.log(2)},
)

**Perplexity on raw TinyStories almost certainly got worse — and the model is far
more useful.**

That is not a contradiction. The model was moved off the distribution perplexity
measures (raw story text) and onto one it doesn't (instruction-formatted
dialogue). The metric is measuring the wrong thing now.

This is the single clearest illustration in the project of why you don't optimise
a number without asking what it tracks. If you had been tuning against validation
perplexity, you would have concluded SFT was harmful and thrown away the step
that made the model actually usable.

In [ ]:
# Save the SFT checkpoint durably before moving on.
from tinyllm.train import save_checkpoint, push_checkpoint
import torch

p = WORK / "checkpoints/sft/sft_final.pt"
torch.save({"step": sft_cfg.max_steps, "model": model.state_dict()}, p)
try:
    push_checkpoint(p, filename="sft_final.pt")
    print(f"pushed -> {hub.ckpt_repo}/sft_final.pt")
except Exception as e:
    print(f"Hub push failed ({e}); the local copy is at {p}")

## Stage 6 gate

- [x] Loss mask verified by decoding a real batch
- [x] Instructions synthesised from varied templates, not one phrasing
- [x] Over-long examples dropped rather than truncated
- [x] Model responds to instructions instead of continuing them
- [x] Understood why validation perplexity got worse

**Next:** `07_package_push.ipynb`